# 馬券風 次単語予想モード — レースデータの作成

LLM が次の単語を決めるまでを競馬に見立てるための CSV を作ります。

- **出走馬** = 次の単語の候補
- **着順** = 最終出力確率の順位
- **途中経過** = 各層で logit lens を掛けた確率（1層 = 100m）

出力した `races.csv` をアプリの設定画面「コンテンツ」→ 馬券風 次単語予想 で取り込み、
そのあとオッズ平均・分散を GUI で調整します。**このノートブックで初期値まで入れる**ので、
そのままでも遊べます。

## 動かす場所

**ローカルでも Colab でも動きます。GPU は要りません**（CPU で十分な大きさのモデルを使います）。

- ローカル … [notebooks/README.md](README.md) の手順で仮想環境を作ってから開く
- Colab … そのまま実行（次のセルが自動で必要なものを入れます）

## 1. 準備

In [ ]:
# Colab のときだけ入れる。ローカルは requirements.txt で入れてある前提
import importlib.util, sys

# find_spec("google.colab") は google パッケージ自体が無いと例外を投げるので、素直に import で判定する
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install "torch>=2.2" "transformers>=4.44,<5" pandas matplotlib

missing = [m for m in ("torch", "transformers", "pandas", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"{', '.join(missing)} がありません。notebooks/README.md の手順で入れてください。")
print("Python", sys.version.split()[0], "on", "Colab" if IN_COLAB else "ローカル")

In [ ]:
import math, json
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "/ numpy", np.__version__)

def setup_japanese_font():
    """グラフの日本語が豆腐（□）にならないようにする。

    matplotlib の既定フォントには日本語が入っていないので、OS にあるものを探して使う。
    見つからなければ諦めて英字だけにする（グラフ自体は出る）。
    """
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("Hiragino Sans", "Hiragino Maru Gothic Pro", "Yu Gothic", "Meiryo",
                 "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic",
                 "IPAGothic", "TakaoGothic", "VL Gothic"):
        if name in have:
            matplotlib.rcParams["font.family"] = name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return name
    return None

FONT = setup_japanese_font()
if FONT:
    print("日本語フォント:", FONT)
else:
    print("日本語フォントが見つかりません。グラフの日本語は □ になります。")
    if IN_COLAB:
        print("  → !apt-get -qq install fonts-ipafont-gothic を実行してランタイムを再起動すると直ります")

## 2. モデル

**CPU で動く大きさ**を選びます。層数がそのままレース距離になります（1層 = 100m）。

| モデル | 層数 | 距離 | メモリ(fp32) | CPU での1レース |
| --- | --- | --- | --- | --- |
| `llm-jp/llm-jp-3-150m` | 12 | 1200m | 約 0.7GB | 数秒 |
| `llm-jp/llm-jp-3-440m` | 16 | 1600m | 約 1.8GB | 十数秒 |
| `llm-jp/llm-jp-3-1.8b` | 24 | 2400m | 約 7GB | 1分前後 |

**既定は 150m** です。まず一通り通してから、余裕があれば大きいものに変えてください。
1.8b はメモリを 8GB 近く使うので、16GB の PC でぎりぎりです。

初回はモデルのダウンロードが入ります（150m で約 0.6GB）。
2 回目からは `~/.cache/huggingface` から読むだけなので速いです。

In [ ]:
MODEL = "llm-jp/llm-jp-3-150m"   # 上の表から選ぶ
N_HORSES = 12                     # 出走頭数（8〜18。アプリ側の制限）
METERS_PER_LAYER = 100            # アプリ側の既定と合わせる

# GPU があれば使うが、無くてよい。mps（Apple Silicon）は層ごとの誤差が出ることがあるので既定は CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
except TypeError:
    # transformers 4.56 より前は dtype ではなく torch_dtype
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
model = model.to(DEVICE).eval()

n_layers = model.config.num_hidden_layers
print(f"{MODEL}  {n_layers}層 → {n_layers * METERS_PER_LAYER}m  ({DEVICE})")

## 3. logit lens

各層の隠れ状態を**最終層の LayerNorm と出力埋め込みに通して**、その層の時点での
「次の単語の確率」を見ます。層 0 は入力そのもので意味がないので 1 層目から使います。

In [ ]:
@torch.no_grad()
def layer_probs(prompt: str, k: int = N_HORSES):
    """最終確率の上位 k 語について、各層での確率を返す。

    戻り値: (語のリスト, 最終確率 [k], 層ごとの確率 [k, 層数])
    """
    ids = {k2: v.to(DEVICE) for k2, v in tok(prompt, return_tensors="pt").items()}
    out = model(**ids, output_hidden_states=True)

    # 最終出力（着順の根拠）
    final = torch.softmax(out.logits[0, -1].float(), dim=-1)
    top = torch.topk(final, k)
    idx = top.indices
    words = [tok.decode([i]) for i in idx.tolist()]

    # 各層を出力空間へ射影する。norm の名前はモデルによって違うので拾いに行く
    base = model.model if hasattr(model, "model") else model.transformer
    norm = getattr(base, "norm", None) or getattr(base, "ln_f", None)
    head = model.get_output_embeddings()

    per_layer = []
    for h in out.hidden_states[1:]:            # 1層目から
        z = h[0, -1].float()
        if norm is not None:
            z = norm(z.to(next(norm.parameters()).dtype)).float()
        logits = head(z.to(head.weight.dtype)).float()
        p = torch.softmax(logits, dim=-1)[idx]
        per_layer.append(p.cpu().numpy())

    return words, top.values.cpu().numpy(), np.stack(per_layer, axis=1)   # [k, 層数]

## 4. 順位変動を見る

**縦軸 logit** と**縦軸 probability** の 2 枚を出します。
順位がコロコロ入れ替わりすぎるレースはここで気づけるので、
プロンプトを変えるか、頭数を減らす判断ができます。

アプリ側でも平滑化は掛かりますが、**元データが暴れすぎているとレースになりません**。

In [ ]:
def plot_race(words, probs, title=""):
    """probs: [頭, 層]"""
    T = probs.shape[1]
    x = np.arange(1, T + 1) * METERS_PER_LAYER
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, (vals, name) in zip(axes, [(np.log(probs + 1e-12), "logit (log prob)"), (probs, "probability")]):
        for i, w in enumerate(words):
            ax.plot(x, vals[i], label=w, linewidth=1.6)
        ax.set_xlabel("distance [m]  (1 layer = %dm)" % METERS_PER_LAYER)
        ax.set_ylabel(name)
        ax.grid(alpha=.3)
    axes[1].legend(fontsize=8, ncol=2, loc="upper left")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 5. オッズの初期値

手で 1 語ずつ入れるのは大変なので、ここで初期値を作ります。

```
oddsMean = clamp(0.80 / 最終確率, 1.1, 500)    ← 控除率 20% ぶんだけ甘い
oddsVar  = (0.18 × oddsMean)²                  ← 変動係数 18%
```

**上位人気が実際に来やすい**設定です。荒れさせたいレースだけ、アプリの GUI で分散を上げてください。

In [ ]:
TAKEOUT = 0.80      # 単勝の控除率
CV = 0.18           # オッズの変動係数（大きいほど荒れる）

def initial_odds(p):
    mean = float(np.clip(TAKEOUT / max(p, 1e-4), 1.1, 500))
    return mean, (CV * mean) ** 2

## 6. レースを作る

プロンプトを並べて実行します。5 レースぶんが既定です。

In [ ]:
PROMPTS = [
    ("R1", "今日は"),
    ("R2", "大学の研究室で"),
    ("R3", "人工知能は"),
    ("R4", "東京の天気は"),
    ("R5", "この問題の答えは"),
]

rows = []
for race_id, prompt in PROMPTS:
    words, final, per_layer = layer_probs(prompt)
    plot_race(words, per_layer, f"{race_id}  {prompt}")
    for i, w in enumerate(words):
        mean, var = initial_odds(float(final[i]))
        row = {
            "race_id": race_id,
            "race_name": f"第{race_id[1:]}R 「{prompt}」",
            "prompt": prompt,
            "model": MODEL,
            "word": w,
            "final_prob": float(final[i]),
            "odds_mean": round(mean, 1),
            "odds_var": round(var, 2),
        }
        for t in range(per_layer.shape[1]):
            row[f"layer_{t+1}"] = float(per_layer[i, t])
        rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head(12)

## 7. 書き出し

ローカルならノートブックと同じ場所に `races.csv` ができます。
アプリの設定画面の「races.csv を取り込む…」でそのファイルを選んでください。

In [ ]:
import os

OUT = "races.csv"
df.to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"{os.path.abspath(OUT)} に {len(df)} 行を書き出しました（{df.race_id.nunique()} レース）")

if IN_COLAB:
    from google.colab import files
    files.download(OUT)

## 補足：荒れ具合の調整

| やりたいこと | 触るところ |
| --- | --- |
| 人気どおりに決まりやすくする | `CV` を下げる（0.10 など） |
| 荒れさせる | `CV` を上げる（0.30 など）。または GUI で特定の語の分散だけ上げる |
| レースを長くする | 層の多いモデルにする。`METERS_PER_LAYER` は見た目の距離が変わるだけ |
| 途中の入れ替わりを増やす | プロンプトを曖昧なものにする（続きが一意に決まらないほど暴れる） |

着順は**最終出力確率だけ**で決まります。途中経過をどういじっても結果は変わりません。